# Part 9 — Advanced Retrieval Patterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-09-advanced-retrieval-patterns/rag_parent_document.ipynb)

*Search on something small and sharp; hand the model something large and rich. They do not have to be the same text.*

📖 Read the essay: https://www.mefby.com/essays/advanced-retrieval-patterns  
🐍 Companion script: `rag_parent_document.py`

For eight parts we sharpened retrieval from every angle — better queries, better candidates, better ranking. But we never questioned the one thing every pipeline quietly assumed: that a single piece of text should do triple duty as **the thing we embed**, **the thing we search**, and **the thing we hand the model**. This part names that assumption and breaks it. The whole chapter rests on one idea — **decoupling the unit you search from the unit you generate from** — and the runnable code below makes the purest pattern, *parent-document retrieval*, real.

**What you'll build**

1. A tiny corpus of real prose (not pre-split chunks) — three policy *parents*.
2. A child/parent split: cut each parent into single-sentence *children*, and keep a `child_to_parent` map.
3. A child scorer — the real Part 6 cosine-over-embeddings search when a model is available, a transparent keyword-overlap fallback otherwise.
4. `retrieve_small_return_big`: find the best **child**, return its **parent** ("index children, serve parents").
5. The before/after demo: what a naive *return-what-you-matched* search hands the model versus parent-document.
6. A peek at the *sentence-window* cousin and *auto-merging* — the de-duplication our code already seeds.

> **Runs fully offline.** Everything here needs only `numpy`. If a real embedding model is cached it is used automatically (the headline path); otherwise the notebook drops to a deterministic, dependency-free stand-in and says so. No network, no API key, ever.

Neighbours: previous is **Part 8 — Making Retrieval Smarter**; next is **Part 10 — Advanced RAG Architectures**.

## Setup

One import block. `re` for the naive sentence splitter, `collections.Counter` for the keyword-overlap fallback scorer, and `numpy` for the cosine math on the real path. Nothing else is required to run this notebook.

In [ ]:
import re
from collections import Counter
import numpy as np

print('imports ready — numpy', np.__version__)

## Step 0 — Start from prose, not pre-split chunks

Every earlier part handed us a neat list of chunks. But *what the chunk should be* is the whole question of this chapter, so we start from the raw thing instead: short sections of a returns policy. Each top-level string is a **parent** — a coherent section, the unit generation actually wants to read.

> **Parent chunk:** a larger unit of text (paragraph, section, document) that gives the model enough context to answer well. It is what we *return*, never what we search.

In [ ]:
# Each string is a PARENT: a coherent section, the unit generation wants to read.
PARENTS = [
    # parent 0
    "Refunds. We accept refunds within 30 days of purchase, as long as the item "
    "is unused and in its original packaging. To start a return, email "
    "support@example.com with your order number. Once we receive the item, your "
    "refund is processed back to the original payment method within five business "
    "days. Shipping fees are not refundable.",
    # parent 1
    "Exchanges. If you want a different size or color, request an exchange instead "
    "of a refund. Exchanges ship free of charge. Items marked final sale cannot be "
    "returned or exchanged. Gift cards are non-refundable and cannot be exchanged.",
    # parent 2
    "Warranty. All electronics include a one-year limited warranty that covers "
    "manufacturing defects. The warranty does not cover accidental damage or normal "
    "wear. To make a warranty claim, contact support with a photo of the defect and "
    "your order number.",
]

for i, p in enumerate(PARENTS):
    print(f'parent {i}: {p[:60]}...')

## Step 1 — Split each parent into children, and remember where they came from

Recall the chunk-size dilemma from Part 5. **Small chunks retrieve precisely** — one vector is about one idea, so it matches sharply. But a lone sentence is too thin to *generate* from. **Large chunks generate well** but match fuzzily, because one vector now averages five ideas. Part 5 treated this as a dial to tune. The insight here is that it was never one dial: it was two jobs wearing one mask.

So we split *twice*. Here a **child** is a single sentence — small, focused, sharp to match. The only new state is `child_to_parent`: a list as long as `children` where entry *i* is the index of the parent that child *i* was carved from. That one list is the entire bookkeeping of parent-document retrieval.

> **Child chunk:** a small unit (a sentence or two) carved from a parent, embedded and searched for sharp matching. A child is never sent to the model alone — it is a pointer to its parent.

In [ ]:
def split_sentences(text):
    # Naive sentence split; good enough for the demo. Use a real splitter for prose.
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

children = []          # the small units we actually embed and search
child_to_parent = []   # child_to_parent[i] = index of the parent child i belongs to
for p_idx, parent_text in enumerate(PARENTS):
    for sentence in split_sentences(parent_text):
        children.append(sentence)
        child_to_parent.append(p_idx)

print(f'{len(PARENTS)} parents  ->  {len(children)} children')

Let's look at the map directly. Each child knows exactly one parent; many children point at the same parent. That many-to-one relationship is what lets a sharp hit on a thin sentence be *traded up* for the rich section it lives in.

In [ ]:
for i, (child, p) in enumerate(zip(children, child_to_parent)):
    print(f'child {i:>2} -> parent {p}   | {child[:54]}')

## Step 2 — Score the children (real model is the headline; fallback keeps it runnable)

Now we need to score each **child** against the query. In your real app this is exactly Part 6's cosine-over-embeddings search — embed the children once, embed the query, take dot products of unit vectors. That is the path you'd actually ship, so it stays the headline:

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')        # 384-dim dense vectors
child_vecs = model.encode(children, normalize_embeddings=True)
q = model.encode([query], normalize_embeddings=True)[0]
scores = list(child_vecs @ q)                          # cosine sim, unit length
```

We wrap that load in a `try/except` so the notebook runs with **no model and no network**. If the import or the weights are unavailable, we drop to a transparent keyword-overlap scorer (next cell). Whichever path runs, it scores **children, never parents** — and the parent/child lesson is identical either way.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def load_real_child_scorer(children):
    """Return a child-scoring fn backed by the real model, or None if unavailable offline."""
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dimensions
        child_vecs = model.encode(children, normalize_embeddings=True)

        def child_scores(query):
            q = model.encode([query], normalize_embeddings=True)[0]
            return list(child_vecs @ q)                    # cosine sim (Part 3)

        return child_scores
    except Exception as exc:  # missing package, no network, no cached weights...
        print(f"[real model unavailable: {type(exc).__name__}] -> offline fallback")
        return None

### The transparent fallback

A real embedding model learned, from billions of sentences, to place similar meanings near each other. We cannot reproduce that without the model. What we *can* do, with zero dependencies, is keep the **same interface** — query in, one score per child out — using a deterministic keyword-overlap count, length-normalised so longer sentences are not unfairly favoured. It does not understand synonyms, but it makes the parent/child mechanics runnable and the numbers reproducible anywhere.

In [ ]:
def make_fallback_child_scorer(children):
    def child_scores(query):
        q = Counter(tokenize(query))
        out = []
        for child in children:
            terms = Counter(tokenize(child))
            overlap = sum(min(q[t], terms[t]) for t in q)
            out.append(overlap / (len(terms) + 1))   # length-normalized overlap
        return out
    return child_scores

Now wire them together: try the real scorer, fall back to keyword overlap. We print which one is live so the output is never ambiguous about what produced it.

In [ ]:
child_scores = load_real_child_scorer(children)
using_real = child_scores is not None
if not using_real:
    child_scores = make_fallback_child_scorer(children)

label = 'REAL all-MiniLM-L6-v2 embeddings' if using_real else 'offline keyword-overlap fallback'
print(f'Child scorer in use: {label}')

## Step 3 — Parent-document retrieval: find the best child, return its parent

Here is the whole pattern. Score the children, take the top `k_children`, then **follow each winner back to its parent through the map** — de-duplicated, so a parent appears at most once. That de-duplication is also the seed of *auto-merging* retrieval: when several top children point at the same parent, returning it once is exactly the merge step.

> **Parent-document retrieval:** embed and search small child chunks, but on a hit return the larger parent chunk they belong to. *Index children, serve parents.*

In [ ]:
def retrieve_small_return_big(query, k_children=3):
    scores = child_scores(query)
    top = sorted(range(len(children)), key=lambda i: scores[i], reverse=True)[:k_children]

    best_child = top[0]
    # De-duplicate parents while preserving the order the children ranked in.
    parent_ids = list(dict.fromkeys(child_to_parent[i] for i in top))

    return {
        "matched_child": children[best_child],                 # what we SEARCHED and hit
        "child_score": float(scores[best_child]),
        "returned_parents": [PARENTS[p] for p in parent_ids],  # what the LLM GETS
    }

print('retrieve_small_return_big defined')

## Step 4 — See the decoupling

Run a query whose sharp match is a *thin* sentence, and watch the before/after of what the model actually receives. We use `k_children=1` to keep the contrast crisp: one sharp child, one rich parent.

- **Naive** (search small, *return small*): hand the model the exact sentence we matched.
- **Parent-document** (search small, *return big*): hand the model the whole parent section that sentence lives in.

In [ ]:
query = "can I get my money back if the box is already open?"
result = retrieve_small_return_big(query, k_children=1)

print(f'Query: {query!r}\n')

print('NAIVE  (search small, return small)  ->  the LLM receives:')
print(f"  [{result['child_score']:.3f}] {result['matched_child']}\n")

print('PARENT-DOCUMENT  (search small, return big)  ->  the LLM receives:')
for parent in result['returned_parents']:
    print(f'  {parent}\n')

print('Same sharp match. Far more context to answer from.')

Read what just happened. The naive path matched a single *true* sentence — and starved the model: it never learns about the 30-day window or the "unused and in original packaging" condition that actually decides this question. Parent-document hands over the whole refund section. **Same sharp match; far more to answer from.**

And note the robustness: swap in the real model and a *different* child may win (semantics beats keyword overlap), but it still lives inside parent 0, so the parent that comes back is identical. **Which child you search is an implementation detail; which parent you serve is the answer.**

## Step 5 — A taste of auto-merging (the de-dup at work)

Raise `k_children` and the de-duplication earns its keep. **Auto-merging retrieval** is parent-document with a quorum: a single stray hit returns just its small chunk, but several hits from the same section are strong evidence the whole section is relevant, so you merge them up. Our `dict.fromkeys` step is exactly that merge — let's watch which parents the top children resolve to as we widen the net.

In [ ]:
q2 = "what are the rules for returning an item I bought?"
scores = child_scores(q2)
ranked = sorted(range(len(children)), key=lambda i: scores[i], reverse=True)

for k in (1, 3, 5):
    top = ranked[:k]
    parents = list(dict.fromkeys(child_to_parent[i] for i in top))
    print(f'k_children={k}: {k} child hits  ->  {len(parents)} parent(s) served: {parents}')

More child hits collapse into a small set of parents — the *amount* of evidence is deciding how much context comes back, which is the whole idea behind auto-merging.

## Step 6 — The close cousin: sentence-window retrieval

Parent-document returns a **fixed, pre-defined unit** — whichever section the child was carved from, with boundaries decided at indexing time. Its cousin, **sentence-window**, returns a **dynamic window centred on the hit**, computed at query time, and it does not care about section boundaries: if the hit is the last sentence of one section, the window happily reaches into the next.

- Reach for **parent-document** when documents have meaningful structure (policy sections, API methods, clauses).
- Reach for **sentence-window** when the right context is *whatever is physically near* the hit — flowing prose, transcripts, articles where ideas bleed across your boundaries.

Here is the same small-to-search idea with a window expansion instead of a parent lookup. We search the *same* children, but on a hit we return the matched sentence plus its `n` neighbours **across parent boundaries**.

> **Sentence-window retrieval:** embed and retrieve single sentences, but on a hit return the matched sentence plus a window of N neighbouring sentences from the original document.

In [ ]:
def retrieve_sentence_window(query, n=1):
    scores = child_scores(query)
    hit = max(range(len(children)), key=lambda i: scores[i])
    lo = max(0, hit - n)
    hi = min(len(children), hit + n + 1)
    return {
        "matched_child": children[hit],
        "window": children[lo:hi],          # neighbours, ignoring section boundaries
        "crossed_boundary": len({child_to_parent[i] for i in range(lo, hi)}) > 1,
    }

win = retrieve_sentence_window("how long do I have to ask for a refund?", n=1)
print('matched child :', win['matched_child'])
print('window (hit ± 1 neighbour):')
for s in win['window']:
    print('   -', s)
print('window crossed a section boundary?', win['crossed_boundary'])

Same sharp match, a *different* notion of "big": parent-document grows to the structural section, while sentence-window grows to physical neighbours — and, depending on where the hit lands, may spill across a section boundary that parent-document would respect. Two answers to the same question, *small to search, big to generate, how exactly?*

## What you learned

- **The core move is decoupling.** The optimal unit to *search* on (small, focused, sharp) is rarely the optimal unit to *generate* from (large, context-rich). Separate them and the Part 5 chunk-size dilemma dissolves.
- **Parent-document is pure bookkeeping.** One list, `child_to_parent`, turns a sharp hit on a thin sentence into the rich section it belongs to. *Index children, serve parents.*
- **The de-dup step seeds auto-merging.** Returning each parent once means the *amount* of evidence decides how much context comes back.
- **Sentence-window is the cousin** that returns proximity instead of structure — a dynamic window that can cross boundaries parent-document respects.
- **The matched child is an implementation detail; the parent you serve is the answer.** Swapping the keyword fallback for a real model can change which child wins, but the served parent is the same.

Two more patterns the essay covers — **self-querying** (an LLM turns a sentence into a semantic query *plus* a metadata filter) and **contextual compression** (trim each retrieved chunk down to the query-relevant content) — stack right on top of this. None of these are free, and none are defaults: add one when failure analysis points at the specific problem it solves, then confirm it helped by measuring (Part 11).

## Next

Our pipeline is sharper than ever, but it is still **static and linear**: one fixed path from query to answer, every time, with no way to ask "did that retrieval actually work?" **Part 10 — Advanced RAG Architectures** breaks that open: agentic RAG, Corrective RAG (CRAG), Self-RAG, GraphRAG, and multi-modal RAG, where the system stops being a conveyor belt and starts being a problem-solver.

📖 https://www.mefby.com/essays/advanced-retrieval-patterns